# Initialization

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Reading From Bronze

In [0]:
df= spark.table("social_media.bronze.badges")

# Data Transformation

## Renaming Columns To Friendly Names

In [0]:
RENAME = {
    "id":"badge_id",
    "user_id":"user_id",
    "name":"badge_name",
    "date":"awarded_date",
    "class":"badge_class",
    "tag_based":"is_tag_based",
    "ingested_at":"ingested_at"
}
for old_name, new_name in RENAME.items():
    df= df.withColumnRenamed(old_name, new_name)

## DataType Casting

In [0]:
df= df.select(
    df["badge_id"].cast(LongType()),
    df["user_id"].cast(LongType()),
    df["badge_name"],
    df["awarded_date"].cast(DateType()),
    df["badge_class"].cast(IntegerType()),
    df["is_tag_based"].cast(BooleanType()),
    df["ingested_at"]
)

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df= df.withColumn(field.name,trim(col(field.name)))

## Data Normalization

In [0]:
df= df.withColumn("badge_class",
                  when(col("badge_class") == 1, "gold").
                  when(col("badge_class") == 2, "silver").
                  otherwise("bronze"))

## Data Consistency

In [0]:
df= df.withColumn("badge_name",lower(regexp_replace(col("badge_name"),"-"," ")))

## Handling Redundancy

In [0]:
df= df.dropDuplicates()

## Sanity Check For DataFrame

In [0]:
df.limit(25).display()

# Writing To Silver

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("social_media.silver.badges")

## Sanity Check For Silver Table

In [0]:
%sql
select * from social_media.silver.badges limit 25;